In [3]:
!pip install -q lightgbm scipy

In [4]:
import warnings
warnings.filterwarnings("ignore")

import os
import sys
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    RepeatedStratifiedKFold
)
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import average_precision_score
from scipy.stats import wilcoxon
from lightgbm import LGBMClassifier

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/Ghana_Dropout_Project"
RESULT_DIR = f"{PROJECT_DIR}/results"
os.makedirs(RESULT_DIR, exist_ok=True)

# Import shared focal loss
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

from losses import (
    focal_loss_lgb,
    focal_loss_eval,
    predict_proba_focal,
    GAMMA, ALPHA, EPSILON
)

print(f"Focal loss config: GAMMA={GAMMA}, ALPHA={ALPHA}")

Mounted at /content/drive
Focal loss config: GAMMA=2.0, ALPHA=0.75


In [5]:
# Load the RAW cleaned data (before feature engineering)
# This is the output of Notebook 1, before Notebook 3 applied
# imputation/encoding on the full dataset.
df = pd.read_csv(f"{PROJECT_DIR}/cleaned_data.csv")
print("Raw cleaned data shape:", df.shape)

TARGET = "dropout_label"
print(df[TARGET].value_counts())
print(f"Dropout rate: {100 * df[TARGET].mean():.1f}%")

Raw cleaned data shape: (1000, 41)
dropout_label
0    908
1     92
Name: count, dtype: int64
Dropout rate: 9.2%


In [6]:
SHARED_PARAMS = dict(
    n_estimators=300,
    num_leaves=31,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    verbosity=-1
)

In [7]:
# ============================================================
# COLUMN CONFIGURATION — match your actual column names
# These must match what Notebook 3 uses.
# ============================================================

ATTENDANCE_COLS = [
    "term_1_attendance",
    "term_2_attendance",
    "term_3_attendance"
]

SOCIOECONOMIC_COLS = {
    "leap_beneficiary": "leap_beneficiary_status",
    "school_feeding": "school_feeding_status",
    "family_income": "family_income_level"
}

BEHAVIOR_COLS = {
    "behavior_warnings": "behaviour_warnings_punishments",
    "class_participation": "class_participation",
    "extracurricular": "extracurricular_activities"
}

# Columns to drop (identifiers, leakage, metadata)
DROP_KEYWORDS = ["id", "student_id", "study_id", "record_id", "serial",
                 "index", "registration", "date_recorded", "enumerator_initials"]
LEAKAGE_COLS = ["dropout_date", "completion_date", "graduation_date",
                "status_after_program", "final_result",
                "headteacher_confirmation_date"]

In [8]:
def preprocess_inside_fold(df_train, df_val, target_col):
    """
    Applies ALL preprocessing and feature engineering using ONLY
    the training fold's statistics. Returns (X_train, y_train, X_val, y_val)
    with composite features included.

    This is the Cause 11 fix: every .fit() and .fit_transform() is
    called on df_train only, then .transform() is applied to df_val.
    """
    train = df_train.copy()
    val = df_val.copy()

    # --- Step 1: Drop identifiers, leakage, empty cols ---
    for d in [train, val]:
        cols_to_drop = [c for c in d.columns
                        if any(k in c.lower() for k in DROP_KEYWORDS)]
        cols_to_drop += [c for c in LEAKAGE_COLS if c in d.columns]
        d.drop(columns=[c for c in cols_to_drop if c in d.columns],
               inplace=True, errors='ignore')

    # --- Step 2: Convert target ---
    label_map = {"0 - Retained": 0, "1 - Dropout": 1}
    for d in [train, val]:
        if d[target_col].dtype == object:
            d[target_col] = d[target_col].astype(str).str.strip().map(label_map)

    # --- Step 3: Yes/No encoding ---
    yes_no_map = {"yes": 1, "no": 0}
    for col in train.columns:
        if train[col].dtype == object:
            vals = train[col].dropna().astype(str).str.lower().unique()
            if set(vals) <= {"yes", "no"}:
                for d in [train, val]:
                    d[col] = d[col].astype(str).str.lower().map(yes_no_map)

    # --- Step 4: Drop constant/near-constant (from train stats only) ---
    train = train.dropna(axis=1, how='all')
    val = val[[c for c in val.columns if c in train.columns]]

    const_cols = [c for c in train.columns if c != target_col and train[c].nunique() <= 1]
    train.drop(columns=const_cols, inplace=True, errors='ignore')
    val.drop(columns=const_cols, inplace=True, errors='ignore')

    # --- Step 5: Separate X/y ---
    y_train = train[target_col].copy()
    y_val = val[target_col].copy()
    X_train = train.drop(columns=[target_col])
    X_val = val.drop(columns=[target_col])

    # --- Step 6: Impute using TRAIN statistics only ---
    numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

    # Median imputation (train stats)
    train_medians = X_train[numeric_cols].median()
    X_train[numeric_cols] = X_train[numeric_cols].fillna(train_medians)
    X_val[numeric_cols] = X_val[numeric_cols].fillna(train_medians)

    # Mode imputation (train stats)
    train_modes = {}
    for col in categorical_cols:
        mode_val = X_train[col].mode()
        train_modes[col] = mode_val[0] if len(mode_val) > 0 else "unknown"
        X_train[col] = X_train[col].fillna(train_modes[col])
        X_val[col] = X_val[col].fillna(train_modes[col])

    # --- Step 7: Label encode categoricals (fit on train only) ---
    label_encoders = {}
    for col in categorical_cols:
        if col in X_train.columns:
            le = LabelEncoder()
            X_train[col] = le.fit_transform(X_train[col].astype(str))
            # Handle unseen categories in val
            val_vals = X_val[col].astype(str)
            known = set(le.classes_)
            val_vals = val_vals.map(lambda x: x if x in known else le.classes_[0])
            X_val[col] = le.transform(val_vals)
            label_encoders[col] = le

    # --- Step 8: Build composite features (using train stats only) ---
    for d in [X_train, X_val]:
        # Attendance risk index
        att_cols_present = [c for c in ATTENDANCE_COLS if c in d.columns]
        if len(att_cols_present) == 3:
            weights = np.array([0.25, 0.30, 0.45])
            att_matrix = d[att_cols_present].to_numpy(dtype=float)
            if np.nanmax(att_matrix) > 1.0:
                att_matrix = att_matrix / 100.0
            risk = 1.0 - att_matrix
            d["attendance_risk_index"] = np.average(risk, axis=1, weights=weights)

    # Socioeconomic vulnerability (uses train stats for income scaling)
    leap_col = SOCIOECONOMIC_COLS["leap_beneficiary"]
    feeding_col = SOCIOECONOMIC_COLS["school_feeding"]
    income_col = SOCIOECONOMIC_COLS["family_income"]

    if all(c in X_train.columns for c in [leap_col, feeding_col, income_col]):
        # income_col is already label-encoded at this point, so use train max
        train_income_max = X_train[income_col].max()
        if train_income_max > 0:
            for d in [X_train, X_val]:
                income_vuln = 1.0 - (d[income_col].astype(float) / train_income_max)
                d["socioeconomic_vulnerability_score"] = (
                    d[leap_col].astype(float) + d[feeding_col].astype(float) + income_vuln
                ) / 3.0

    # Behavioral engagement (MinMaxScaler fit on train only)
    warn_col = BEHAVIOR_COLS["behavior_warnings"]
    part_col = BEHAVIOR_COLS["class_participation"]
    extra_col = BEHAVIOR_COLS["extracurricular"]

    behav_cols = [warn_col, part_col, extra_col]
    if all(c in X_train.columns for c in behav_cols):
        scaler = MinMaxScaler()
        train_behav = X_train[behav_cols].astype(float)
        val_behav = X_val[behav_cols].astype(float)

        train_scaled = scaler.fit_transform(train_behav)
        val_scaled = scaler.transform(val_behav)

        for d, scaled in [(X_train, train_scaled), (X_val, val_scaled)]:
            d["behavioral_engagement_index"] = (
                (1.0 - scaled[:, 0]) + scaled[:, 1] + scaled[:, 2]
            ) / 3.0

    # --- Step 9: Drop low-variance (from train stats) ---
    low_var = [c for c in X_train.columns
               if c not in ["attendance_risk_index",
                            "socioeconomic_vulnerability_score",
                            "behavioral_engagement_index"]
               and X_train[c].value_counts(normalize=True).iloc[0] > 0.99]
    X_train.drop(columns=low_var, inplace=True, errors='ignore')
    X_val = X_val[[c for c in X_val.columns if c in X_train.columns]]

    return X_train, y_train, X_val, y_val

In [9]:
# Quick test: does the preprocessing function work?
test_train, test_val = train_test_split(df, test_size=0.2, random_state=42,
                                         stratify=df[TARGET])
Xt, yt, Xv, yv = preprocess_inside_fold(test_train, test_val, TARGET)
print(f"Train: {Xt.shape}, Val: {Xv.shape}")
print(f"Columns: {Xt.columns.tolist()[:10]}...")
print(f"Composites present: {[c for c in Xt.columns if c in ['attendance_risk_index', 'socioeconomic_vulnerability_score', 'behavioral_engagement_index']]}")
print(f"Any NaNs in train? {Xt.isnull().sum().sum()}")
print(f"Any NaNs in val? {Xv.isnull().sum().sum()}")

Train: (800, 43), Val: (200, 43)
Columns: ['school_code', 'geographic_zone', 'school_type', 'data_source', 'gender', 'age_at_start_of_academic_year', 'class_level', 'grade_repetition_count', 'term_1_attendance', 'term_2_attendance']...
Composites present: ['attendance_risk_index', 'socioeconomic_vulnerability_score', 'behavioral_engagement_index']
Any NaNs in train? 0
Any NaNs in val? 0


In [10]:
SEEDS = [42, 123, 456, 789, 1024, 2048, 3333, 5555, 7777, 9999]
N_SPLITS = 5
N_REPEATS = 5

all_seed_results = []

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*60}")
    print(f"SEED {seed} ({seed_idx+1}/{len(SEEDS)})")
    print(f"{'='*60}")

    rskf = RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        random_state=seed
    )

    baseline_scores = []
    focal_scores = []

    fold_num = 0
    for train_idx, val_idx in rskf.split(df, df[TARGET]):
        fold_num += 1

        df_train_fold = df.iloc[train_idx]
        df_val_fold = df.iloc[val_idx]

        # Cause 11 fix: preprocess INSIDE the fold
        X_tr, y_tr, X_vl, y_vl = preprocess_inside_fold(
            df_train_fold, df_val_fold, TARGET
        )

        # Identify composite vs raw features
        composite_feats = [c for c in [
            "attendance_risk_index",
            "socioeconomic_vulnerability_score",
            "behavioral_engagement_index"
        ] if c in X_tr.columns]

        raw_feats = [c for c in X_tr.columns if c not in composite_feats]

        # Baseline: raw features only, default cross-entropy
        baseline_model = LGBMClassifier(
            objective="binary",
            is_unbalance=True,
            random_state=seed,
            **SHARED_PARAMS
        )
        baseline_model.fit(X_tr[raw_feats], y_tr)
        baseline_prob = baseline_model.predict_proba(X_vl[raw_feats])[:, 1]
        baseline_auc = average_precision_score(y_vl, baseline_prob)
        baseline_scores.append(baseline_auc)

        # E-LightGBM: all features + focal loss
        focal_model = LGBMClassifier(
            objective=focal_loss_lgb,
            random_state=seed,
            **SHARED_PARAMS
        )
        focal_model.fit(X_tr, y_tr, eval_metric=focal_loss_eval)
        focal_prob = predict_proba_focal(focal_model, X_vl)
        focal_auc = average_precision_score(y_vl, focal_prob)
        focal_scores.append(focal_auc)

    baseline_arr = np.array(baseline_scores)
    focal_arr = np.array(focal_scores)
    diff = focal_arr - baseline_arr

    stat, pval = wilcoxon(focal_arr, baseline_arr)

    pooled_std = np.sqrt((baseline_arr.std(ddof=1)**2 + focal_arr.std(ddof=1)**2) / 2)
    d = diff.mean() / pooled_std if pooled_std > 0 else float('nan')

    result = {
        'Seed': seed,
        'Baseline_Mean': baseline_arr.mean(),
        'Baseline_SD': baseline_arr.std(ddof=1),
        'ELightGBM_Mean': focal_arr.mean(),
        'ELightGBM_SD': focal_arr.std(ddof=1),
        'Diff_Mean': diff.mean(),
        'Diff_Sign': '+' if diff.mean() > 0 else '-',
        'Wilcoxon_p': pval,
        'Cohens_d': d
    }
    all_seed_results.append(result)

    print(f"  Baseline: {baseline_arr.mean():.4f} +/- {baseline_arr.std(ddof=1):.4f}")
    print(f"  E-LightGBM: {focal_arr.mean():.4f} +/- {focal_arr.std(ddof=1):.4f}")
    print(f"  Diff: {diff.mean():+.4f} (sign: {result['Diff_Sign']})")
    print(f"  Wilcoxon p = {pval:.4f}, Cohen's d = {d:.4f}")


SEED 42 (1/10)
  Baseline: 0.9869 +/- 0.0198
  E-LightGBM: 0.9842 +/- 0.0196
  Diff: -0.0027 (sign: -)
  Wilcoxon p = 0.0302, Cohen's d = -0.1360

SEED 123 (2/10)
  Baseline: 0.9835 +/- 0.0314
  E-LightGBM: 0.9827 +/- 0.0278
  Diff: -0.0008 (sign: -)
  Wilcoxon p = 0.1712, Cohen's d = -0.0266

SEED 456 (3/10)
  Baseline: 0.9895 +/- 0.0234
  E-LightGBM: 0.9864 +/- 0.0235
  Diff: -0.0030 (sign: -)
  Wilcoxon p = 0.0786, Cohen's d = -0.1297

SEED 789 (4/10)
  Baseline: 0.9911 +/- 0.0147
  E-LightGBM: 0.9913 +/- 0.0132
  Diff: +0.0002 (sign: +)
  Wilcoxon p = 0.4068, Cohen's d = 0.0168

SEED 1024 (5/10)
  Baseline: 0.9869 +/- 0.0218
  E-LightGBM: 0.9844 +/- 0.0238
  Diff: -0.0025 (sign: -)
  Wilcoxon p = 0.0520, Cohen's d = -0.1091

SEED 2048 (6/10)
  Baseline: 0.9843 +/- 0.0259
  E-LightGBM: 0.9827 +/- 0.0279
  Diff: -0.0016 (sign: -)
  Wilcoxon p = 0.4938, Cohen's d = -0.0591

SEED 3333 (7/10)
  Baseline: 0.9818 +/- 0.0303
  E-LightGBM: 0.9804 +/- 0.0311
  Diff: -0.0014 (sign: -)
  Wilc

In [11]:
# ============================================================
# RESULTS TABLE: 10-Seed Stability
# ============================================================

seed_df = pd.DataFrame(all_seed_results)
display(seed_df)

seed_df.to_csv(f"{RESULT_DIR}/diagnostic_10_seed_stability.csv", index=False)

# Key diagnostic checks
n_positive = (seed_df['Diff_Sign'] == '+').sum()
n_negative = (seed_df['Diff_Sign'] == '-').sum()

print(f"\n{'='*60}")
print(f"CAUSE 9 VERDICT")
print(f"{'='*60}")
print(f"Seeds where E-LightGBM WINS:  {n_positive}/10")
print(f"Seeds where Baseline WINS:    {n_negative}/10")
print(f"Mean diff across all seeds:   {seed_df['Diff_Mean'].mean():+.4f}")
print(f"SD of diff across seeds:      {seed_df['Diff_Mean'].std(ddof=1):.4f}")

if n_positive >= 4 and n_negative >= 4:
    print("\n>>> SIGN FLIPS across seeds. The original result was NOISE.")
    print(">>> The negative result is NOT stable. Cause 9 CONFIRMED.")
elif n_negative >= 8:
    print("\n>>> Sign is STABLE negative. The baseline consistently wins.")
    print(">>> Cause 9 is CLEARED (sign stable). Proceed to other diagnostics.")
elif n_positive >= 8:
    print("\n>>> Sign is STABLE positive after fixing Cause 11!")
    print(">>> The negative result was caused by the pre-split leakage.")
    print(">>> E-LightGBM ACTUALLY WINS when preprocessing is done correctly.")
else:
    print(f"\n>>> Mixed results ({n_positive} positive, {n_negative} negative).")
    print(">>> Inconclusive. Report as such.")

,Seed,Baseline_Mean,Baseline_SD,ELightGBM_Mean,ELightGBM_SD,Diff_Mean,Diff_Sign,Wilcoxon_p,Cohens_d
0,42,0.986922,0.019846,0.984239,0.019614,-0.002683,-,0.030221,-0.135979
1,123,0.983468,0.031363,0.982680,0.027811,-0.000788,-,0.171195,-0.026597
2,456,0.989450,0.023387,0.986409,0.023488,-0.003041,-,0.078644,-0.129736
3,789,0.991104,0.014742,0.991340,0.013201,0.000235,+,0.406777,0.016822
4,1024,0.986886,0.021779,0.984399,0.023775,-0.002487,-,0.052036,-0.109106
5,2048,0.984261,0.025883,0.982670,0.027942,-0.001591,-,0.493814,-0.059084
6,3333,0.981766,0.030273,0.980372,0.031085,-0.001394,-,0.231059,-0.045440
7,5555,0.988104,0.019481,0.982949,0.021301,-0.005154,-,0.005107,-0.252532
8,7777,0.987292,0.019912,0.987088,0.019532,-0.000204,-,0.711942,-0.010331
9,9999,0.985659,0.026630,0.984911,0.026622,-0.000749,-,0.443585,-0.028115



CAUSE 9 VERDICT
Seeds where E-LightGBM WINS:  1/10
Seeds where Baseline WINS:    9/10
Mean diff across all seeds:   -0.0018
SD of diff across seeds:      0.0016

>>> Sign is STABLE negative. The baseline consistently wins.
>>> Cause 9 is CLEARED (sign stable). Proceed to other diagnostics.


In [12]:
# Power analysis for paired Wilcoxon test
# Using the normal approximation for paired differences

from scipy.stats import norm

# Observed values from the 10-seed run
observed_diffs = seed_df['Diff_Mean'].values
observed_mean_diff = observed_diffs.mean()
observed_sd_diff = observed_diffs.std(ddof=1)

# Per-fold SD (use the pooled across all seeds)
# Approximate using the average within-seed SD of the difference
n_folds = N_SPLITS * N_REPEATS  # 25

# MDE for a paired t-test (approximation for Wilcoxon)
alpha = 0.05
power = 0.80
z_alpha = norm.ppf(1 - alpha/2)
z_beta = norm.ppf(power)

# We need SD of per-fold differences. Use the average from all seeds.
# Each seed has 25 fold-level differences.
avg_baseline_sd = seed_df['Baseline_SD'].mean()
avg_focal_sd = seed_df['ELightGBM_SD'].mean()
avg_pooled_sd = np.sqrt((avg_baseline_sd**2 + avg_focal_sd**2) / 2)

MDE = (z_alpha + z_beta) * avg_pooled_sd / np.sqrt(n_folds)

print(f"{'='*60}")
print(f"CAUSE 17: POWER ANALYSIS")
print(f"{'='*60}")
print(f"Number of paired observations (folds): {n_folds}")
print(f"Average pooled SD of AUC-PR:           {avg_pooled_sd:.4f}")
print(f"Significance level (alpha):            {alpha}")
print(f"Desired power:                         {power}")
print(f"")
print(f"Minimum Detectable Effect (MDE):       {MDE:.4f}")
print(f"Observed mean difference:              {observed_mean_diff:+.4f}")
print(f"")

if abs(observed_mean_diff) < MDE:
    print(f">>> |Observed diff| ({abs(observed_mean_diff):.4f}) < MDE ({MDE:.4f})")
    print(f">>> Study is UNDERPOWERED to detect this effect size.")
    print(f">>> The null result should be labelled INCONCLUSIVE, not 'no effect'.")
else:
    print(f">>> |Observed diff| ({abs(observed_mean_diff):.4f}) >= MDE ({MDE:.4f})")
    print(f">>> Study has adequate power. The null is informative.")

# Save
power_df = pd.DataFrame([{
    'N_Folds': n_folds,
    'Alpha': alpha,
    'Power': power,
    'Avg_Pooled_SD': avg_pooled_sd,
    'MDE': MDE,
    'Observed_Mean_Diff': observed_mean_diff,
    'Underpowered': abs(observed_mean_diff) < MDE
}])
power_df.to_csv(f"{RESULT_DIR}/diagnostic_power_analysis.csv", index=False)
display(power_df)

CAUSE 17: POWER ANALYSIS
Number of paired observations (folds): 25
Average pooled SD of AUC-PR:           0.0234
Significance level (alpha):            0.05
Desired power:                         0.8

Minimum Detectable Effect (MDE):       0.0131
Observed mean difference:              -0.0018

>>> |Observed diff| (0.0018) < MDE (0.0131)
>>> Study is UNDERPOWERED to detect this effect size.
>>> The null result should be labelled INCONCLUSIVE, not 'no effect'.


,N_Folds,Alpha,Power,Avg_Pooled_SD,MDE,Observed_Mean_Diff,Underpowered
0,25,0.05,0.8,0.023383,0.013102,-0.001786,True


In [13]:
print(f"{'='*60}")
print("LEAVE-ONE-SEED-OUT STABILITY")
print(f"{'='*60}")

full_mean = seed_df['Diff_Mean'].mean()
full_sign = '+' if full_mean > 0 else '-'
print(f"Full aggregate mean diff: {full_mean:+.4f} (sign: {full_sign})")
print()

loo_results = []
for i, row in seed_df.iterrows():
    remaining = seed_df.drop(i)
    loo_mean = remaining['Diff_Mean'].mean()
    loo_sign = '+' if loo_mean > 0 else '-'
    flipped = loo_sign != full_sign

    loo_results.append({
        'Removed_Seed': row['Seed'],
        'Remaining_Mean_Diff': loo_mean,
        'Sign': loo_sign,
        'Flipped': flipped
    })

    status = 'FLIP!' if flipped else 'stable'
    print(f"  Remove seed {int(row['Seed']):>5d}: mean diff = {loo_mean:+.4f} ({status})")

loo_df = pd.DataFrame(loo_results)
n_flips = loo_df['Flipped'].sum()

print(f"\nFlips: {n_flips}/{len(SEEDS)}")
if n_flips == 0:
    print(">>> Conclusion is ROBUST to removing any single seed.")
else:
    print(f">>> Conclusion FLIPS when {n_flips} seed(s) are removed.")
    print(">>> One or more seeds are carrying the average.")

loo_df.to_csv(f"{RESULT_DIR}/diagnostic_leave_one_seed_out.csv", index=False)

LEAVE-ONE-SEED-OUT STABILITY
Full aggregate mean diff: -0.0018 (sign: -)

  Remove seed    42: mean diff = -0.0017 (stable)
  Remove seed   123: mean diff = -0.0019 (stable)
  Remove seed   456: mean diff = -0.0016 (stable)
  Remove seed   789: mean diff = -0.0020 (stable)
  Remove seed  1024: mean diff = -0.0017 (stable)
  Remove seed  2048: mean diff = -0.0018 (stable)
  Remove seed  3333: mean diff = -0.0018 (stable)
  Remove seed  5555: mean diff = -0.0014 (stable)
  Remove seed  7777: mean diff = -0.0020 (stable)
  Remove seed  9999: mean diff = -0.0019 (stable)

Flips: 0/10
>>> Conclusion is ROBUST to removing any single seed.
